**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB05 — Representations: Where Identity and Context Information Sit

**Track B · Mechanistic interpretability of SEL-4L · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | After the embedding, where in the network (layer, position) is identity information present; how far does a change of player spread; and at which depth does the context-versus-identity preference form? |
| **Experiments** | E9 propagation of a player change, E10 linear probes with control tasks, E11 logit lens |
| **Evidence level** | representational only: that information is present or decodable does not show that the model uses it. Causal claims start in MB06. |
| **Outputs** | `results/MB05_*`, `figures/MB05_*` |
| **Module written** | `src/probing.py` |
| **Test data** | none |



**Probe interpretation.** The existing `control_labels` function remaps each true target class to a random class. It can preserve target information, so the legacy “selectivity” column is only an accuracy difference against a target-class remapping baseline; it is not independent control-task selectivity. Some logistic regressions reach the existing 300-iteration limit; warnings are retained below. Probe accuracy is a decoding diagnostic and does not establish causal use. These limitations do not affect the activation collection or later patching experiments.

In [1]:
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = "4"
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /home/user/code/soccer_mi
Track A (read only): /home/user/code/soccer_mi/model_training_evaluation | exists: True
Track B: /home/user/code/soccer_mi/.mi_analysis_rerun


In [2]:
# Shared source is versioned with this notebook; do not overwrite repaired dependencies.
assert (MI_DIR / 'src/probing.py').exists(), "Missing dependency: src/probing.py"

In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.probing as probing
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, probing]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Note: no GPU found. The full run works on CPU but is much slower.


Device: cpu | outputs: /home/user/code/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [4]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(val_conflicts.groupby(["population", "half"]).size().unstack().to_string())

partners = pd.read_parquet(POP / "swap_partners.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
population = val_conflicts[val_conflicts[IDENTITY] > 0].reset_index(drop=True)
POINTS = ["resid_pre_0"] + [f"resid_post_{l}" for l in range(N_LAYERS)]
print(f"identity population: {len(population):,} rows | residual points: {POINTS}")

/home/user/code/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 48 | primary seed 103 | identity variable: player
half        discovery  estimation
population                       
P-player         1215         529
index-0          4296        4651
identity population: 1,744 rows | residual points: ['resid_pre_0', 'resid_post_0', 'resid_post_1', 'resid_post_2', 'resid_post_3']


## 1. Terms

| Term | Meaning here |
|---|---|
| **Representation** | A pattern in the activations that varies systematically with some variable (for example, which player acts). |
| **Probe** | A small classifier trained to read a variable from activations. |
| **Linear probe** | A probe that is a single linear layer (here multinomial logistic regression). Success means the information is present in an easily readable form. |
| **Control task** | The same probe trained on labels re-assigned at random by type (Hewitt & Liang 2019). It measures how much a probe can memorise by itself. |
| **Selectivity** | Probe accuracy minus control-task accuracy: the part of the accuracy that reflects the activations rather than the probe's own capacity. |
| **Logit lens** | Applying the final LayerNorm and the output layer to an intermediate residual stream, to read a provisional prediction at that depth. |

## 2. E9 — How far does a change of player travel?

For each identity-population event, the model is run on the event as recorded and with the player at the target replaced by the matched partner (I2; for a non-player identity variable, removal I1). At each residual point and each position, the size of the difference is divided by the average size of the clean residual there.

**Expected by construction, not a finding:** exactly zero at every position before the target (the causal mask); this is asserted. Positions after the target are shown for completeness; they cannot affect the target's prediction.

**Control:** the norm-matched random vector (I4) — the same measurement for a change of the same size that carries no learned player information.

In addition, the difference at the target is projected onto the Δ direction through the frozen final LayerNorm: how much of the change at each depth points towards a_ctx rather than a_id.

In [5]:
offsets = np.arange(-15, 16)
sums = {kind: np.zeros((len(POINTS), len(offsets))) for kind in ["swap", "vector"]}
counts = np.zeros((len(POINTS), len(offsets)))
projection_rows = []
generator = torch.Generator().manual_seed(0)
with torch.no_grad():
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, population, BATCH):
        chunk = population.iloc[part]
        if IDENTITY == "player":
            partner = torch.as_tensor([partner_of[p] if partner_of.get(p, -1) > 0 else p for p in chunk["player"]], device=DEVICE)
            changed = interventions.set_feature(inputs, "player", positions, partner)
        else:
            changed = interventions.remove_feature(inputs, IDENTITY, positions)
        _, clean = hooks.forward(MODEL, inputs, capture=POINTS + ["final_resid"])
        _, swap = hooks.forward(MODEL, changed, capture=POINTS)
        _, vector = hooks.forward(MODEL, inputs, capture=POINTS,
                                  replace={f"factor_{IDENTITY}": interventions.norm_matched_vector(None, positions, generator)})
        rows = torch.arange(len(positions), device=DEVICE)
        scale = hooks.final_layernorm_scale(MODEL, clean["final_resid"])[rows, positions]
        u = MODEL.unembed.weight[a_ctx] - MODEL.unembed.weight[a_id]
        for p_index, point in enumerate(POINTS):
            clean_norm = clean[point].norm(dim=-1)
            for kind, other in [("swap", swap), ("vector", vector)]:
                relative = ((other[point] - clean[point]).norm(dim=-1) / clean_norm.mean()).cpu().numpy()
                for o_index, offset in enumerate(offsets):
                    key = (positions + int(offset)).cpu().numpy()
                    valid = (key >= 0) & (key < 16)
                    sums[kind][p_index, o_index] += relative[np.where(valid)[0], key[valid]].sum()
                    if kind == "swap":
                        counts[p_index, o_index] += valid.sum()
            difference = (swap[point] - clean[point])[rows, positions]
            centred = difference - difference.mean(dim=-1, keepdim=True)
            projection = ((u * MODEL.ln_final.weight * centred).sum(-1) * scale.squeeze(-1)).cpu().numpy()
            projection_rows.append(pd.DataFrame({"point": point, "projection": projection, "match_id": chunk["match_id"].to_numpy()}))
propagation = {kind: sums[kind] / np.maximum(counts, 1) for kind in sums}
before_target = propagation["swap"][:, offsets < 0]
print("largest relative change at positions before the target:", float(before_target.max()), "(must be exactly 0)")
assert before_target.max() == 0.0

fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
for axis, kind, title in [(axes[0], "swap", "matched player swap"), (axes[1], "vector", "norm-matched random vector (control)")]:
    image = axis.imshow(propagation[kind], aspect="auto", cmap="magma", extent=[offsets[0] - 0.5, offsets[-1] + 0.5, len(POINTS) - 0.5, -0.5])
    axis.set_yticks(range(len(POINTS))); axis.set_yticklabels(POINTS)
    axis.set_xlabel("position relative to the target (0 = target)"); axis.set_title(f"Relative change of the residual stream: {title}")
    plt.colorbar(image, ax=axis, fraction=0.03)
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_propagation.png", bbox_inches="tight"); plt.show()

projection = pd.concat(projection_rows)
projection_table = pd.DataFrame([{"point": point, **mi_metrics.summarise(group["projection"], group["match_id"], N_BOOT)}
                                 for point, group in projection.groupby("point", sort=False)])
print("\nProjection of the swap-induced change at the target onto the delta direction (nats, frozen final LayerNorm):")
print(projection_table[["point", "n", "mean", "ci_low", "ci_high"]].round(4).to_string(index=False))

largest relative change at positions before the target: 0.0 (must be exactly 0)



Projection of the swap-induced change at the target onto the delta direction (nats, frozen final LayerNorm):
       point    n   mean  ci_low  ci_high
 resid_pre_0 1744 0.0081 -0.0070   0.0244
resid_post_0 1744 0.0107 -0.0129   0.0328
resid_post_1 1744 0.0279 -0.0186   0.0678
resid_post_2 1744 0.0473 -0.0173   0.1084
resid_post_3 1744 0.1272 -0.0356   0.2833


/tmp/ipykernel_303942/3490312700.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_propagation.png", bbox_inches="tight"); plt.show()


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 3. E10 — Linear probes with control tasks

**Activations.** The residual stream at the scoring position of each event, at six points (embedding, after each of the 4 blocks, after the final LayerNorm). Probes are fitted on events sampled from the **training** split and tested on **validation** events; the two share no match.

**Targets.**

| Target | Classes | What it tells |
|---|---|---|
| acting player | the 50 most frequent T300 players (other events excluded) | identity content at the target |
| acting team | the 50 most frequent teams | identity content |
| role | 25 roles | identity content |
| previous action | 16 | history content |
| pitch third | 3 | spatial content |
| acting team changed | 2 | possession-state content |
| coupled with previous | 2 | annotation structure |
| previous actor | 50 most frequent T300 players, read at the target | whether the previous event's identity is moved into the target position by attention |

**Probe.** Standardised multinomial logistic regression; the regularisation strength C is chosen from {0.1, 1} by 3-fold cross-validation grouped by match. **Controls:** (a) control task with the same C; (b) majority-class rate; (c) the same probe on a randomly initialised model of SEL-4L's shape (seed 0, untrained). **Metrics:** validation accuracy and selectivity.

**Reading rule, stated before the results.** The embedding point should decode the current player, team and role almost perfectly, because their vectors are added there; the informative quantities are how that changes with depth, and whether the *previous* actor becomes decodable at the target.

**Limit of the control task here.** For variables that are themselves model inputs (player, team, role, previous action), each label *type* has its own input vector, so the control task is nearly as easy as the real task and selectivity is close to zero by construction. Selectivity is informative mainly for the previous actor and for derived variables (pitch third); for the direct inputs, the comparison with the random-init model and the change with depth carry the information.

In [6]:
def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])


rng = np.random.default_rng(0)
train_view = bundle["views"]["train"]
features = {name: tensor.cpu().numpy() for name, tensor in bundle["encoded"].tensors.items()}
def sample_view(view, n):
    chosen = np.sort(rng.choice(len(view["event_rows"]), size=min(n, len(view["event_rows"])), replace=False))
    windows = view["window_index"][chosen]; positions = view["position"][chosen]
    event_rows = view["event_rows"][chosen]
    previous_rows = np.where(positions > 0, view["windows"][windows, np.maximum(positions - 1, 0)], -1)
    return {"windows": windows, "positions": positions, "event_rows": event_rows, "previous_rows": previous_rows}
fit_sample = sample_view(train_view, SETTINGS["probe_train_events"])
test_sample = sample_view(val_view, SETTINGS["probe_train_events"])

train_counts = pd.Series(features["player"][events["split"].to_numpy() == "train"]).value_counts()
top_players = [p for p in train_counts.index if p > 0][:50]
team_counts = pd.Series(features["team"][events["split"].to_numpy() == "train"]).value_counts()
top_teams = [t for t in team_counts.index if t > 0][:50]
third = np.minimum((events["x_norm"].to_numpy() * 3).astype(int), 2)
def labels_for(sample, target):
    rows = sample["event_rows"]
    if target == "acting player":
        values = features["player"][rows]; keep = np.isin(values, top_players)
    elif target == "acting team":
        values = features["team"][rows]; keep = np.isin(values, top_teams)
    elif target == "role":
        values = features["role"][rows]; keep = values > 0
    elif target == "previous action":
        values = features["prev_action"][rows]; keep = np.ones(len(rows), bool)
    elif target == "pitch third":
        values = third[rows]; keep = np.ones(len(rows), bool)
    elif target == "acting team changed":
        values = events["team_change"].to_numpy()[rows]; keep = np.ones(len(rows), bool)
    elif target == "coupled with previous":
        values = events["coupled_with_prev"].to_numpy()[rows]; keep = np.ones(len(rows), bool)
    else:
        previous = sample["previous_rows"]
        values = np.where(previous >= 0, features["player"][np.maximum(previous, 0)], -1)
        keep = np.isin(values, top_players) & (previous >= 0)
    return values, keep
TARGETS = ["acting player", "acting team", "role", "previous action", "pitch third", "acting team changed",
           "coupled with previous", "previous actor"]

random_config = mi_setup.load_json(run_dir(PRIMARY_SEED) / "run_config.json")
torch.manual_seed(0)
random_model = training.build_model(random_config, bundle["vocabulary_sizes"]).to(DEVICE).eval()
probe_points = POINTS + ["final_normalised"]
activations = {}
for name, model in [("SEL-4L", MODEL), ("random init", random_model)]:
    for sample_name, sample in [("fit", fit_sample), ("test", test_sample)]:
        activations[(name, sample_name)] = probing.collect(model, bundle, train_view if sample_name == "fit" else val_view,
                                                           sample["windows"], sample["positions"], probe_points, BATCH, hooks.forward)
print("activations collected:", {k: v[probe_points[0]].shape for k, v in activations.items()})

activations collected: {('SEL-4L', 'fit'): (20000, 128), ('SEL-4L', 'test'): (20000, 128), ('random init', 'fit'): (20000, 128), ('random init', 'test'): (20000, 128)}


In [7]:
match_ids_all = bundle["match_ids"]
probe_rows = []
start = time.time()
for target in TARGETS:
    y_fit_all, keep_fit = labels_for(fit_sample, target)
    y_test_all, keep_test = labels_for(test_sample, target)
    y_fit, y_test = y_fit_all[keep_fit], y_test_all[keep_test]
    if len(np.unique(y_fit)) < 2 or len(y_test) == 0:
        continue
    groups = match_ids_all[fit_sample["event_rows"][keep_fit]]
    control_fit = probing.control_labels(np.concatenate([y_fit, y_test]), seed=0)
    control_test = control_fit[len(y_fit):]; control_fit = control_fit[:len(y_fit)]
    majority = float(pd.Series(y_test).value_counts(normalize=True).iloc[0])
    for point in probe_points:
        print(f"Probe: {target} / {point}; fit n={len(y_fit)}, classes={len(np.unique(y_fit))}", flush=True)
        x_fit = activations[("SEL-4L", "fit")][point][keep_fit]
        x_test = activations[("SEL-4L", "test")][point][keep_test]
        probe, best_c, cv_score = probing.fit_probe(x_fit, y_fit, groups, [0.1, 1.0])
        accuracy = float(probe.score(x_test, y_test))
        control_probe = probing.make_pipeline(probing.StandardScaler(), probing.LogisticRegression(C=best_c, max_iter=300))
        control_probe.fit(x_fit, control_fit)
        control_accuracy = float(control_probe.score(x_test, control_test))
        random_probe = probing.make_pipeline(probing.StandardScaler(), probing.LogisticRegression(C=best_c, max_iter=300))
        random_probe.fit(activations[("random init", "fit")][point][keep_fit], y_fit)
        random_accuracy = float(random_probe.score(activations[("random init", "test")][point][keep_test], y_test))
        probe_rows.append({"target": target, "point": point, "classes": len(np.unique(y_fit)), "n fit": len(y_fit),
                           "n test": len(y_test), "C": best_c, "accuracy": accuracy, "control-task accuracy": control_accuracy,
                           "selectivity": accuracy - control_accuracy, "random-init accuracy": random_accuracy,
                           "majority rate": majority})
        print(f"Completed probe in cumulative {time.time()-start:.1f}s", flush=True)
probes = pd.DataFrame(probe_rows)
probes.to_csv(OUT / "results" / "MB05_probes.csv", index=False)
print(f"{len(probes)} probes in {time.time() - start:.0f} s")
print(probes.round(3).to_string(index=False))

Probe: acting player / resid_pre_0; fit n=2267, classes=50


Completed probe in cumulative 1.1s


Probe: acting player / resid_post_0; fit n=2267, classes=50


Completed probe in cumulative 1.9s


Probe: acting player / resid_post_1; fit n=2267, classes=50


...

Probe: previous actor / resid_post_3; fit n=2228, classes=50


Completed probe in cumulative 279.5s


Probe: previous actor / final_normalised; fit n=2228, classes=50


Completed probe in cumulative 283.1s


48 probes in 283 s
               target            point  classes  n fit  n test   C  accuracy  control-task accuracy  selectivity  random-init accuracy  majority rate
        acting player      resid_pre_0       50   2267    1649 1.0     0.993                  0.962        0.031                 0.990          0.107
        acting player     resid_post_0       50   2267    1649 1.0     0.970                  0.921        0.049                 0.988          0.107
        acting player     resid_post_1       50   2267    1649 1.0     0.944                  0.879        0.064                 0.983          0.107
        acting player     resid_post_2       50   2267    1649 1.0     0.916                  0.859        0.058                 0.984          0.107
        acting player     resid_post_3       50   2267    1649 1.0     0.896                  0.826        0.070                 0.983          0.107
        acting player final_normalised       50   2267    1649 1.0     0.895     

In [8]:
fig, axes = plt.subplots(2, 4, figsize=(20, 7), sharex=True)
for axis, target in zip(axes.flatten(), TARGETS):
    frame = probes[probes.target == target]
    if len(frame) == 0:
        axis.set_visible(False); continue
    x = np.arange(len(frame))
    axis.plot(x, frame["accuracy"], "o-", label="SEL-4L probe")
    axis.plot(x, frame["control-task accuracy"], "s--", label="control task")
    axis.plot(x, frame["random-init accuracy"], "^:", label="random-init model")
    axis.axhline(frame["majority rate"].iloc[0], color="grey", linewidth=0.8, label="majority class")
    axis.set_xticks(x); axis.set_xticklabels([p.replace("resid_", "") for p in frame["point"]], rotation=45, fontsize=7)
    axis.set_title(target); axis.set_ylim(0, 1.02)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_probes.png", bbox_inches="tight"); plt.show()

/tmp/ipykernel_303942/3229017713.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_probes.png", bbox_inches="tight"); plt.show()


**How to read the figure.** One panel per target; the x-axis runs from the embedding to the final LayerNorm. The distance between the solid line and the control-task line is selectivity. A random-init line close to the SEL-4L line means the information is readable from the input construction alone, not from anything learned.

**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

## 4. E11 — Logit lens on Δ

Δ is read from each residual point at the target through the final LayerNorm and output layer. The last point equals the model's Δ (asserted). Reported for the identity population, index-0 rows and matched controls. Limitation stated before the result: intermediate layers are not trained to be read this way, so a flat curve does not show that nothing happens.

In [9]:
def lens(table, label):
    rows = []
    with torch.no_grad():
        for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, table, BATCH):
            logits, cache = hooks.forward(MODEL, inputs, capture=POINTS)
            index = torch.arange(len(positions), device=DEVICE)
            final_delta = mi_metrics.delta(mi_metrics.at_target(logits, positions), a_ctx, a_id)
            for point in POINTS:
                read = MODEL.unembed(MODEL.ln_final(cache[point][index, positions]))
                value = mi_metrics.delta(read, a_ctx, a_id)
                if point == POINTS[-1]:
                    assert np.allclose(value, final_delta, atol=1e-4)
                rows.append(pd.DataFrame({"population": label, "point": point, "delta": value,
                                          "match_id": table.iloc[part]["match_id"].to_numpy()}))
    return pd.concat(rows)
lens_frames = [lens(population, "identity population"), lens(val_conflicts[val_conflicts[IDENTITY] == 0], "index-0")]
if len(controls):
    lens_frames.append(lens(controls, "matched controls"))
lens_all = pd.concat(lens_frames)
lens_table = pd.DataFrame([{"population": pop, "point": point, **mi_metrics.summarise(g["delta"], g["match_id"], N_BOOT)}
                           for (pop, point), g in lens_all.groupby(["population", "point"], sort=False)])
lens_table.to_csv(OUT / "results" / "MB05_logit_lens.csv", index=False)
print(lens_table[["population", "point", "n", "mean", "ci_low", "ci_high"]].round(3).to_string(index=False))
fig, axis = plt.subplots(figsize=(8, 4))
for pop, frame in lens_table.groupby("population", sort=False):
    axis.errorbar(range(len(frame)), frame["mean"], yerr=[frame["mean"] - frame["ci_low"], frame["ci_high"] - frame["mean"]],
                  fmt="o-", label=pop, capsize=3)
axis.set_xticks(range(len(POINTS))); axis.set_xticklabels(POINTS, rotation=20)
axis.set_ylabel("delta read at this depth (nats)"); axis.legend(); axis.set_title("Logit lens: arbitration by depth")
plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_logit_lens.png", bbox_inches="tight"); plt.show()

mi_setup.save_json(OUT / "results" / "MB05_summary.json", {"projection": projection_table.to_dict(orient="records"),
                   "probes": probes.to_dict(orient="records"), "logit_lens": lens_table.to_dict(orient="records"),
                   "propagation_swap": propagation["swap"].tolist(), "propagation_vector": propagation["vector"].tolist(),
                   "offsets": offsets.tolist(), "points": POINTS, "run_mode": RUN_MODE})
fingerprint.record_files(MANIFEST, {"results/MB05_summary.json": OUT / "results" / "MB05_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB05", "experiment": "E9-E11 representations", "identity_variable": IDENTITY,
                                "seed": PRIMARY_SEED, "run_mode": RUN_MODE})

         population        point    n   mean  ci_low  ci_high
identity population  resid_pre_0 1744 -0.381  -0.489   -0.264
identity population resid_post_0 1744  1.363   1.258    1.476
identity population resid_post_1 1744  3.185   3.013    3.365
identity population resid_post_2 1744  3.932   3.789    4.090
identity population resid_post_3 1744  5.963   5.728    6.187
            index-0  resid_pre_0 8947 -0.342  -0.430   -0.242
            index-0 resid_post_0 8947  1.472   1.397    1.544
            index-0 resid_post_1 8947  3.465   3.368    3.559
            index-0 resid_post_2 8947  4.306   4.194    4.417
            index-0 resid_post_3 8947  6.306   6.187    6.427
   matched controls  resid_pre_0  252 -0.481  -0.671   -0.226
   matched controls resid_post_0  252  0.010  -0.241    0.310
   matched controls resid_post_1  252  1.319   0.983    1.750
   matched controls resid_post_2  252  1.766   1.341    2.276
   matched controls resid_post_3  252  2.422   1.876    3.056


/tmp/ipykernel_303942/1561080323.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / "figures" / "MB05_logit_lens.png", bbox_inches="tight"); plt.show()


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):